---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Introduction to Pydantic

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.



---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



Before class, please work through the examples below and try the short exercises. 

We'll use movie reviews, just as we did in the previous lecture. Don't worry about memorizing the syntax of Pydantic — we'll use it again in class.

# 📦 1. Dictionaries are useful, but...

We've been using dictionaries to keep related information together. Here's a movie review:

In [ ]:
review_dict = {
    'title': 'Dune',
    'rating': 'excellent',
}

print(review_dict)

Python is perfectly happy with this dictionary. But what if the rest of our code expects `rating` to be a number?

This is where **Pydantic** helps. We can describe what our data should look like and have Python check it when we create an object.

The package is already in the course's `pyproject.toml`. If you need to update your environment, run `uv sync` in the terminal from your `wa` folder, then select that environment as your notebook kernel.

In [ ]:
from pydantic import BaseModel, Field, ValidationError, model_validator
from typing import Literal, Self


# 🧱 2. Our first Pydantic model

Let's start with just two fields: a title and a numeric rating.

In [ ]:
class Review(BaseModel):
    """A movie title and the reviewer's numeric rating."""

    title: str = Field(description='Title of the movie being reviewed.')
    rating: float = Field(description='Numeric rating given by the reviewer.')


What's happening here?

- `class Review` defines a **class** — think of it as a template for our reviews.
- `(BaseModel)` means `Review` **inherits from** another class, Pydantic's `BaseModel`. It gets validation and useful methods from that parent class, so we don't have to write them ourselves.
- The triple-quoted sentence is a **docstring**: a short explanation of what this class represents.
- The other indented lines describe its **fields**.
- `Field(description=...)` explains what each field means. Writing these descriptions is a good habit!
- `title: str` says the title should be a string; `rating: float` says the rating should be a number.

We haven't covered classes in detail yet. For now, think of inheritance as building on a class that already knows how to do useful things.

The part after the colon is a **type annotation**. These annotations don't normally make Python check values by themselves; Pydantic uses them to do that checking.

Also, “model” here means a description of our data. It is **not** a machine learning model!

The docstring describes the whole model; each field description explains one part. These also help describe the data we want when we use models with LLMs in the next class. Descriptions document our intentions; types and constraints do the checking.


In [ ]:
# Create one review using our new type
review = Review(title='Dune', rating=4.5)

print(review)
print(review.title)
print(review.rating)

We've created an **instance** of `Review`: one particular review.

Notice the dots: `review.title`, rather than `review_dict['title']`. These are attributes, like the ones we discussed in the functions and methods notebook.

We can also create a review from a dictionary:

In [ ]:
review_data = {'title': 'Dune', 'rating': 4.5}
review = Review.model_validate(review_data)

print(review)

# 🚦 3. What happens when the data is wrong?

Let's try the dictionary from the beginning, where the rating was `'excellent'` instead of a number.

In [ ]:
try:
    Review.model_validate(review_dict)
except ValidationError as error:
    print(error)

Read the message: it points to `rating` and explains why Pydantic couldn't turn it into a number.

The `try/except` lets us display the error and continue the notebook. Without it, this cell would stop with a `ValidationError`.

A field without a default is also **required**. Let's leave out the rating:

In [ ]:
try:
    Review(title='Dune')
except ValidationError as error:
    print(error)

# 🔄 4. Sometimes Pydantic converts the type

Not every input with a different type is rejected. If the conversion makes sense, Pydantic may do it for us:

In [ ]:
# This rating starts as a string, but Pydantic can turn it into a float
review = Review(title='Dune', rating='4.5')

print(review.rating)
print(type(review.rating))

So `'4.5'` became the number `4.5`. But `'excellent'` couldn't be converted.

## Your turn

Create a `Review` for a movie of your choice. Try a number, a numeric string, and some text as the rating. Which inputs work, and what type does the resulting rating have? Catch `ValidationError` for the invalid example.

In [ ]:
# Your code here

# 📏 5. Adding rules, choices, and defaults

A numeric rating isn't enough. Our first model will accept this:

In [ ]:
# It is a number, so our basic Review model accepts it
print(Review(title='Dune', rating=99))

Let's write a more specific model. Here, we'll use a rating scale from **1 to 5**.

In [ ]:
class MovieReview(BaseModel):
    """A movie review with a rating from 1 to 5 and a sentiment label."""

    title: str = Field(
        min_length=1,
        description='Title of the movie being reviewed.',
    )
    rating: float = Field(
        ge=1,
        le=5,
        description="Reviewer's rating, from 1 (lowest) to 5 (highest).",
    )
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(
        description="Reviewer's overall sentiment about the movie.",
    )
    reviewer_name: str | None = Field(
        default=None,
        description='Name of the reviewer, if known.',
    )


A few new things:

- `min_length=1`: the title can't be an empty string.
- `ge=1` and `le=5`: greater than or equal to 1, and less than or equal to 5.
- `Literal[...]`: the sentiment must be one of those exact strings.
- `str | None`: the reviewer name can be a string or `None`.
- `default=None` in `Field(...)`: if we leave out the reviewer name, use `None`.

That last distinction matters: allowing `None` and allowing a field to be omitted aren't the same thing. The default is what lets us omit it.

In [ ]:
# We haven't supplied a reviewer name, so it gets the default
review = MovieReview(title='Dune', rating=4.5, sentiment='positive')

print(review)
print('Reviewer:', review.reviewer_name)

Now let's try an out-of-range rating and a sentiment that isn't in our list:

In [ ]:
invalid_reviews = [
    {'title': 'Dune', 'rating': 6, 'sentiment': 'positive'},
    {'title': 'Dune', 'rating': 4.5, 'sentiment': 'mixed'},
]

for data in invalid_reviews:
    try:
        MovieReview.model_validate(data)
    except ValidationError as error:
        print(error)
        print()

Pretty useful, right? We can catch these problems before passing the data to the rest of our code.

One important limit: validation checks the types and rules we've specified. It doesn't check whether someone actually wrote this review, or whether the information is true.

# 📝 6. Dictionaries and JSON

We often want to save our data or pass it to other code. `model_dump()` gives us a regular Python dictionary:

In [ ]:
review_data = review.model_dump()

print(review_data)
print(type(review_data))

And `model_dump_json()` gives us a JSON **string**, not a file. Remember JSON from Lecture 6? Python's `None` becomes `null` in JSON.

In [ ]:
json_text = review.model_dump_json(indent=2)

print(json_text)
print(type(json_text))

We can go the other way too. `model_validate_json()` reads the JSON string, checks it, and creates a model instance:

In [ ]:
restored_review = MovieReview.model_validate_json(json_text)

print(restored_review)
print(restored_review.title)
print(type(restored_review.rating))

# 🎬 7. Objects inside other objects

Let's make something a bit more interesting: a movie page with a title, a release year, and several reviews.

The reviews can be `MovieReview` objects! `list[MovieReview]` says that each item in the list should be one of those objects. We'll require at least one review.


In [ ]:
class MoviePage(BaseModel):
    """A movie's details and the reviews displayed on its page."""

    title: str = Field(
        min_length=1,
        description='Title of the movie this page is about.',
    )
    release_year: int = Field(
        ge=1900,
        description='Year the movie was released.',
    )
    reviews: list[MovieReview] = Field(
        min_length=1,
        description="Reviews displayed on this movie's page.",
    )


We can pass in the `review` object we already created, or a dictionary with the right fields. Pydantic turns the dictionary into a `MovieReview` and checks it too.


In [ ]:
movie_page = MoviePage(
    title='Dune',
    release_year=2021,
    reviews=[
        review,
        {
            'title': 'Dune',
            'rating': 4,
            'sentiment': 'positive',
            'reviewer_name': 'Jordan',
        },
    ],
)

# Pick a review from the list, then access one of its attributes
print(movie_page.reviews[0].rating)
print(movie_page.reviews[1].reviewer_name)
print(type(movie_page.reviews[1]))


So `movie_page.reviews[0].rating` means: get the page's reviews, take the first review, and get its rating.

Here, `MoviePage` **contains** `MovieReview` objects. It also **inherits from** `BaseModel`, just as `MovieReview` does.

What if one of the reviews has an invalid rating?


In [ ]:
# model_dump() also turns the inner reviews into dictionaries
bad_page_data = movie_page.model_dump()
bad_page_data['reviews'][0]['rating'] = 6

try:
    MoviePage.model_validate(bad_page_data)
except ValidationError as error:
    print(error)


The error points to `reviews.0.rating`: the rating in the first review. We didn't have to write another rating check for the bigger object.

JSON works for the whole page too. When we read it back, we get the inner `MovieReview` objects back as well:


In [ ]:
page_json = movie_page.model_dump_json(indent=2)
print(page_json)

restored_page = MoviePage.model_validate_json(page_json)
print(type(restored_page.reviews[0]))


# 🔍 8. A post-validator: checking the whole object

There's one thing we haven't checked: do all these reviews belong to the movie on the page? A review of *Barbie* can be perfectly valid on its own, but it shouldn't appear on our *Dune* page.

Let's add that rule. For this example, the titles must match exactly.

`CheckedMoviePage` inherits from **our own** `MoviePage` class. It gets the same fields and validation, and adds one extra check:


In [ ]:
class CheckedMoviePage(MoviePage):
    """A movie page whose reviews all match its movie title."""

    @model_validator(mode='after')
    def check_review_titles(self) -> Self:
        """Check that every review is for the movie on this page."""
        # Each review has already passed MovieReview's checks
        for review in self.reviews:
            if review.title != self.title:
                raise ValueError('Every review must be for the movie on this page.')
        return self


A few things to unpack:

- `@model_validator(mode='after')` is a **decorator**: a label above the method that tells Pydantic to run it after the fields have passed validation, including the nested reviews. That's why we call it a post-validator.
- `self` is the particular page being checked. We can use both `self.title` and `self.reviews`, so this check can compare different parts of the object.
- `raise ValueError(...)` rejects the page. Pydantic includes our message in a `ValidationError`.
- `return self` passes the valid page back. Don't forget it! `-> Self` is a type annotation saying that the method returns an instance of this class.

Pydantic calls this method for us when we create or validate a `CheckedMoviePage`. We don't call it ourselves.


In [ ]:
# All reviews are for Dune, so this works
checked_page = CheckedMoviePage.model_validate(movie_page.model_dump())
print(checked_page.title)
print('Number of reviews:', len(checked_page.reviews))


Now let's keep the rating and sentiment valid, but make the first review belong to another movie:


In [ ]:
wrong_movie_data = movie_page.model_dump()
wrong_movie_data['reviews'][0]['title'] = 'Barbie'

try:
    CheckedMoviePage.model_validate(wrong_movie_data)
except ValidationError as error:
    print(error)


The individual fields passed, but our extra rule didn't. That's the point of a model validator: we can check how the parts fit together.


# 🛠 9. Your turn: a movie model

Create a Pydantic model called `Movie` with these fields:

- `title`: a string with at least one character.
- `release_year`: an integer. For this exercise, require a year of at least 1900.
- `genres`: a list of strings, written `list[str]`.
- `director`: a string or `None`, defaulting to `None`.

Add a short class docstring and a `Field(description=...)` for each field.

Create one valid movie and print its title. Then try invalid data and read the validation error. Finally, convert your valid movie to JSON and use your model to read it back.

What happens if you leave out `director`? What happens if you leave out `title`?

In [ ]:
# Your code here

# Next class

We'll use models like these to describe the structured data we want from an LLM in [Using LLMs](10.2%20using-llms.ipynb).

If you'd like to look things up:

- [Pydantic: models](https://docs.pydantic.dev/latest/concepts/models/)
- [Pydantic: fields and constraints](https://docs.pydantic.dev/latest/concepts/fields/)
- [Pydantic: serialization](https://docs.pydantic.dev/latest/concepts/serialization/)
- [Pydantic: custom validators](https://docs.pydantic.dev/latest/concepts/validators/#model-validators)
